# SMS Spam Detection — ML Lab

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style('whitegrid')

import re
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

In [ ]:
url = "https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"
df = pd.read_csv(url, encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df['message_length'] = df['message'].apply(len)

plt.figure(figsize=(6,4))
sns.countplot(x='label', data=df, palette=['#4C72B0', '#DD8452'])
plt.title('Class Distribution')
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(data=df, x='message_length', hue='label', bins=50, kde=True, palette=['#4C72B0', '#DD8452'])
plt.title('Message Length by Class')
plt.show()

In [ ]:
ham_text = " ".join(df[df['label'] == 'ham']['message'])
spam_text = " ".join(df[df['label'] == 'spam']['message'])
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(WordCloud(width=600, height=400, background_color='white', colormap='Blues').generate(ham_text), interpolation='bilinear')
axes[0].set_title('Ham Words')
axes[0].axis('off')
axes[1].imshow(WordCloud(width=600, height=400, background_color='white', colormap='Reds').generate(spam_text), interpolation='bilinear')
axes[1].set_title('Spam Words')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]
    return " ".join(words)

df['clean_message'] = df['message'].apply(clean_text)

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['clean_message']).toarray()
le = LabelEncoder()
y = le.fit_transform(df['label'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear', probability=True),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

results = []
trained_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })
    trained_models[name] = model
    predictions[name] = y_pred

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
results_df

In [ ]:
best_model_name = results_df.iloc[0]['Model']
cm = confusion_matrix(y_test, predictions[best_model_name])
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title(f'Confusion Matrix — {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()
print(classification_report(y_test, predictions[best_model_name], target_names=['Ham', 'Spam']))

In [ ]:
def predict_message(text):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned]).toarray()
    pred = trained_models[best_model_name].predict(vec)[0]
    return le.inverse_transform([pred])[0]

sample_messages = [
    "Congratulations! You've won a $1000 Walmart gift card. Click here to claim now!!!",
    "Hey, are we still on for lunch tomorrow at 1pm?",
    "URGENT: Your bank account has been suspended. Verify your details immediately.",
    "Don't forget to bring your laptop to class tomorrow.",
    "FREE entry into our weekly competition, text WIN to 80086 now!"
]

for msg in sample_messages:
    print(f"[{predict_message(msg).upper():5}]  {msg}")